# Ollama en SageMaker (m5.xlarge)
1. Crear la *Lifecycle configuration* **on-start** con el contenido de `sagemaker/on-start.sh` y asociarla al notebook.
2. Crear el notebook instance `ml.m5.xlarge` (volumen ≥ 30 GB) y abrir JupyterLab.
3. Ejecutar estas celdas: verifican Docker, `data-root`, el contenedor, el modelo y una respuesta real.

Todo lo persistente vive en `/home/ec2-user/SageMaker`.

In [1]:
!docker info 2>/dev/null | grep "Docker Root Dir"   # debe ser /home/ec2-user/SageMaker/docker-data
!docker ps --filter name=ollama

 Docker Root Dir: /home/ec2-user/SageMaker/docker-data


CONTAINER ID   IMAGE                  COMMAND               CREATED        STATUS       PORTS                                           NAMES
1209f522da18   ollama/ollama:latest   "/bin/ollama serve"   25 hours ago   Up 2 hours   0.0.0.0:11434->11434/tcp, :::11434->11434/tcp   ollama


Si es la primera vez y la lifecycle aún no corrió, se puede lanzar a mano:

In [2]:
# !sudo bash /home/ec2-user/SageMaker/LabDino/sagemaker/on-start.sh

In [3]:
MODEL = "gemma4:e2b"
!docker exec ollama ollama pull {MODEL}
!docker exec ollama ollama list

pulling manifest ⠋ 

pulling manifest ⠙ 

pulling manifest ⠹ pulling manifest 
pulling 6dd7ac24fe13: 100% ▕██████████████████▏ 3.5 GB                         
pulling 7ba3bbacdf1b: 100% ▕██████████████████▏ 986 MB                         
pulling dd57b3b7308f: 100% ▕██████████████████▏  97 MB                         
pulling b507b9c2f6ca: 100% ▕██████████████████▏   13 B                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 2365fbb6d97b: 100% ▕██████████████████▏   64 B                         
pulling 3ff8a734c316: 100% ▕██████████████████▏  241 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


NAME          ID              SIZE      MODIFIED               
gemma4:e2b    b37049369adf    4.6 GB    Less than a second ago    


In [4]:
import requests, json
BASE = "http://localhost:11434"
print("Versión de Ollama:", requests.get(f"{BASE}/api/version").json())
info = requests.post(f"{BASE}/api/show", json={"model": MODEL}).json()
print(json.dumps(info.get("details"), indent=2))

Versión de Ollama: {'version': '0.40.8'}


{
  "parent_model": "gemma4-validation-20260929:e2b-imatrix-attention-q6-q8-draft2",
  "format": "gguf",
  "family": "gemma4",
  "families": [
    "gemma4"
  ],
  "parameter_size": "4.6B",
  "quantization_level": "Q4_K_M",
  "runner": "llamacpp"
}


### Prueba: el modelo recibe una instrucción y responde

In [5]:
r = requests.post(f"{BASE}/api/generate", json={"model": MODEL, "prompt": "Di hola en una frase como un dinosaurio.",
                                                "stream": False, "think": False}, timeout=600)
print(r.json()["response"])

¡**Grrr**, hola! 🦖


### Registrar modelo y versión, y generar la identidad con el código del repo
Clonar el repo dentro de `/home/ec2-user/SageMaker` (persistente). Solo se necesita `httpx` y `PYTHONPATH=src`. `generate_identity.py` genera la ficha del nombre elegido (doractodon) con Ollama de SageMaker y escribe `ollama/identity.json` e `identity.md`.

In [6]:
%cd /home/ec2-user/SageMaker
!test -d LabDino || git clone https://github.com/Ooojulian/LabDino
%cd LabDino
!pip install -q httpx
import os
os.environ["OLLAMA_BASE_URL"] = BASE
os.environ["OLLAMA_MODEL"] = MODEL
os.environ["PYTHONPATH"] = "src"
!python ollama/write_model_info.py
!python ollama/generate_identity.py

/home/ec2-user/SageMaker


/home/ec2-user/SageMaker/LabDino


# Modelo de Ollama utilizado

Generado por `ollama/write_model_info.py` a partir de la API de Ollama.

| Campo | Valor |
|---|---|
| Modelo (nombre exacto) | gemma4:e2b |
| Digest | b37049369adfe3d2b653af0ab301a062ca5cbe96ace6aa0d3f2559ee9b563fc2 |
| Tamaño en disco (bytes) | 4587156505 |
| Familia | gemma4 |
| Parámetros | 4.6B |
| Cuantización | Q4_K_M |
| Formato | gguf |
| Capacidades | completion, vision, audio, tools, thinking |
| Versión de Ollama | 0.40.8 |
| Registrado (UTC) | 2026-10-23 02:46:54 |



# Doractodon

*Ficha ficticia generada con `gemma4:e2b` (Ollama 0.40.8) el 2026-10-23 02:54:55 UTC; temperature 0.7, seed 44, think activado, 3 intento(s), 480.5 s.*

## Apariencia

Doractodon presenta un cuerpo robusto y bajo, con una estructura pesada adaptada al terreno pantanoso. Posee una piel gruesa de color verde oliva oscuro que le proporciona camuflaje en el sotobosque. Sus extremidades son cortas y fuertes, diseñadas para la locomoción lenta y estable.

## Habitat

Vive en pantanos tropicales con vegetación densa y exuberante. El clima es húmedo y cálido, con abundante agua estancada y suelos anegados.

## Alimentacion

Consume plantas fibrosas y raíces mediante una mandíbula potente. La dentición especializada, compuesta por dientes cónicos, permite desgarrar la vegetación densa con gran eficiencia.

## Comportamiento

El dinosaurio es predominantemente solitario, buscando refugio en áreas boscosas durante el día. Exhibe movimientos cautelosos y evita el contacto con depreda

### Exponer Ollama a la app web (ngrok)
Guardar el token **fuera del repo**: `echo TU_TOKEN > /home/ec2-user/SageMaker/.ngrok_token` (la lifecycle lo arranca en cada inicio).
La URL pública se lee aquí y va en `OLLAMA_BASE_URL` del backend, con `OLLAMA_NGROK_SKIP_WARNING=true`.

In [7]:
import requests
try:
    tunnels = requests.get("http://localhost:4040/api/tunnels").json()["tunnels"]
    print([t["public_url"] for t in tunnels])
except Exception as e:
    print("ngrok no está activo:", e)

ngrok no está activo: HTTPConnectionPool(host='localhost', port=4040): Max retries exceeded with url: /api/tunnels (Caused by NewConnectionError("HTTPConnection(host='localhost', port=4040): Failed to establish a new connection: [Errno 111] Connection refused"))
